# Breast Cancer Prediction Pipeline

This notebook implements the main pipeline for breast cancer classification using RNA-Seq data. It includes data loading and preprocessing, statistical feature selection (ANOVA F-test), dimensionality reduction (PCA), and classification using a Random Forest model.

## Setup and Configuration

This step sets up the required libraries and defines the folder structure used in this notebook. The data is organized into four main folders under `data/`:
- `raw/`: Raw downloaded files (e.g., GTEx `.gct`, GDC `.tsv`).
- `initial/`: Processed raw files (e.g., merged and transposed).
- `interim/`: Feature-selected or partially transformed files.
- `processed/`: Final model-ready inputs (e.g., PCA-reduced features).

In [1]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
import shutil
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.metrics import roc_curve, auc

In [2]:
# Define data folder structure
DATA_DIR = "data"
RAW_GDC_DIR = os.path.join(DATA_DIR, "raw_gdc_data")
RAW_GTX_DIR = os.path.join(DATA_DIR, "raw_gtx_data")

INITIAL_DIR = os.path.join(DATA_DIR, "initial")
INTERIM_DIR = os.path.join(DATA_DIR, "interim")

MODELS_DIR = "models"
REPORTS_DIR = "reports"

In [ ]:
# Create directories if they don't exist
os.makedirs(INITIAL_DIR, exist_ok=True)
os.makedirs(INTERIM_DIR, exist_ok=True)

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

### Step 1: Initial Process of GDC Raw Sample Files

This block processes RNA-Seq files downloaded from the GDC portal. Each `.tsv` file contains gene expression data for a single cancer sample. From each file, the column `tpm_unstranded` is extracted, indexed by `gene_id`.

These per-sample files are merged into a single matrix where:
- **Rows** = genes,
- **Columns** = samples,
- **Values** = TPM expression values.

This process results in a file `gdc_data.csv` saved to the `data/initial/` folder. 

It is computationally expensive, so it is designed to be skipped once completed unless the raw data changes.

#### Step 1.1: Merge All Sample Files
Merging separate cancer sample files and Building a complete dataset file

In [ ]:
# Define paths for GDC sample sheet and data files
gdc_sample_sheet_path = os.path.join(RAW_GDC_DIR, 'gdc_sample_sheet.tsv')
gdc_merged_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

In [ ]:
sample_sheet_file_handler = pd.read_csv(gdc_sample_sheet_path, sep='\t')

# Initialize an empty DataFrame to store the combined data
gdc_df = pd.DataFrame()

sample_counter = 1
# Iterate through each file listed in the sample sheet
for index, row in sample_sheet_file_handler.iterrows():
    folder_id = row['File ID']
    file_name = row['File Name']
    
    # Construct the full file path
    sample_file_path = os.path.join(RAW_GDC_DIR, folder_id, file_name)
    
    # Read the TSV file
    try:
        sample_data = pd.read_csv(sample_file_path, sep='\t', skiprows=[0, 2, 3, 4, 5])
        
        # Extract the relevant columns ('gene_id' and 'TPM' or equivalent)
        relevant_data = sample_data[['gene_id', 'tpm_unstranded']]
        
        # Rename the columns to match the GTex format
        sample_number = 'c_' + str(sample_counter).zfill(4)
        relevant_data.columns = ['gene_id', sample_number]  # Use folder_id as the sample name
        
        # Merge with the combined data
        if gdc_df.empty:
            gdc_df = relevant_data
        else:
            gdc_df = pd.merge(gdc_df, relevant_data, on='gene_id', how='outer')

        sample_counter += 1
            
    except Exception as e:
        print(f"Error processing file {sample_file_path}: {e}")

gdc_df.to_csv(gdc_merged_file_path, index=False)

In [ ]:
# Load and display the first few rows of the merged GDC dataset
gdc_merged_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

gdc_merged_file_df = pd.read_csv(gdc_merged_file_path, index_col=0)
print("Shape of initial GDC Data Set", gdc_merged_file_df.shape)
gdc_merged_file_df.head(5)

#### Step 1.2: Split into Model Training Data and Unseen Testing Data
Building a model training dataset of `train_num` samples and an unseen dataset of `unseen_num` samples for testing fo deploying model.

Default Values of GDC tcga data for Breast Cancer: 
- Total number of samples: `total_num = 1231`
- Model Training `train_num= 1000`
- Unseen Testing Data `unseen_num = 231`

In [3]:
# Define path
gdc_data_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

gdc_df = pd.read_csv(gdc_data_file_path, nrows=5)
columns = gdc_df.columns.tolist()

gene_info_columns = columns[0:1]
sample_columns = columns[1:]

In [ ]:
# Split data into data for model building and training and unseen data for testing
gdc_total_num = len(sample_columns)         # bc tcga data = 1231
gdc_train_num = 1000
gdc_unseen_num = gdc_total_num - gdc_train_num

In [ ]:
# Randomly select 1000 sample columns from the dataset as traiing data
gdc_chosen_columns_for_training = np.random.choice(sample_columns, gdc_train_num, replace=False).tolist()
gdc_final_columns_for_training = gene_info_columns + gdc_chosen_columns_for_training

# Get the remaining columns as testing data
gdc_chosen_columns_for_testing = [col for col in sample_columns if col not in gdc_chosen_columns_for_training]
gdc_final_columns_for_testing = gene_info_columns + gdc_chosen_columns_for_testing

In [ ]:
# Load the dataset again but only with the selected columns
gdc_train_df = pd.read_csv(gdc_data_file_path, usecols=gdc_final_columns_for_training)
gdc_unseen_df = pd.read_csv(gdc_data_file_path, usecols=gdc_final_columns_for_testing)

In [ ]:
# Save the training and testing data to new files
gdc_training_file_path = os.path.join(INITIAL_DIR, 'gdc_data_training.csv')
gdc_testing_file_path = os.path.join(INITIAL_DIR, 'gdc_data_testing.csv')

gdc_train_df.to_csv(gdc_training_file_path, index=False)
gdc_unseen_df.to_csv(gdc_testing_file_path, index=False)

In [8]:
print("Shape of initial GDC Training Data Set", gdc_train_df.shape)
gdc_train_df.head(5)

Shape of initial GDC Training Data Set (60660, 1001)


,gene_id,c_0001,c_0002,c_0004,c_0005,c_0006,c_0007,c_0008,c_0009,c_0010,...,c_1220,c_1221,c_1222,c_1224,c_1225,c_1226,c_1227,c_1228,c_1229,c_1230
0,ENSG00000000003.15,49.6341,12.0296,26.5679,18.0535,10.6064,24.0318,63.9724,75.6881,11.2357,...,84.4328,52.3403,27.3544,30.1448,26.2866,37.7670,85.6117,71.7311,5.3684,113.1190
1,ENSG00000000005.6,9.3826,0.3785,4.0213,0.3523,0.6661,1.3877,0.3128,0.3828,0.0000,...,12.1832,0.2608,0.0979,0.0567,8.9343,0.0000,0.3046,0.6878,0.3944,0.4400
2,ENSG00000000419.13,115.1737,134.9047,86.0077,71.4260,112.2490,52.8140,125.0148,111.9771,66.6285,...,111.7066,107.1986,82.4554,126.5208,63.2222,165.8215,148.1734,115.8308,129.9716,82.5384
3,ENSG00000000457.14,20.1202,15.5758,19.7490,10.7816,3.3119,21.4261,15.2346,8.9836,3.8821,...,22.8129,16.1887,13.8403,29.8414,11.5421,11.6838,12.4748,16.0663,27.4784,19.6494
4,ENSG00000000460.17,6.1859,4.3777,16.9112,5.0191,3.8493,4.8368,9.3567,15.2681,2.3710,...,4.6271,6.7480,4.8875,12.4256,4.6340,7.6037,7.6431,4.8100,12.2774,8.9295


In [9]:
print("Shape of initial GDC Testing Data Set", gdc_unseen_df.shape)
gdc_unseen_df.head(5)

Shape of initial GDC Testing Data Set (60660, 232)


,gene_id,c_0003,c_0013,c_0019,c_0024,c_0032,c_0045,c_0048,c_0050,c_0062,...,c_1167,c_1192,c_1196,c_1203,c_1204,c_1207,c_1208,c_1209,c_1223,c_1231
0,ENSG00000000003.15,90.4249,58.2760,39.7555,14.1390,28.8024,31.8464,61.9126,95.3834,55.3278,...,16.0181,11.4142,20.5496,29.0778,54.1708,29.2468,26.5100,51.8718,16.6648,11.5596
1,ENSG00000000005.6,0.0000,82.3055,4.8758,0.1385,6.3480,2.3704,0.0407,0.0000,42.2069,...,0.0000,0.7400,0.2161,0.0000,0.3428,1.9849,0.2804,0.1706,0.1169,0.0424
2,ENSG00000000419.13,110.8229,90.0477,146.8816,116.7291,232.5009,61.6094,89.7537,200.3726,102.0887,...,95.0445,117.2234,57.3113,196.4417,136.4973,87.4715,82.1935,170.4536,85.6008,283.6449
3,ENSG00000000457.14,31.8373,11.5658,7.6407,16.1782,15.5013,18.4199,8.3279,9.4441,10.4126,...,3.5482,13.5948,5.5746,8.3812,12.8766,12.2471,23.6649,11.6550,12.5955,17.0290
4,ENSG00000000460.17,15.4869,2.7522,5.5405,1.6178,6.9078,9.2477,5.1429,14.0108,3.3701,...,3.5109,6.6965,3.0265,5.7478,10.1232,11.4997,4.6224,4.1838,4.9241,8.8565


### Step 2: Initial Process of GTEx Raw Sample Files

This step processes transcriptomic data from GTEx. The original file is in `.gct` format and includes expression levels for thousands of genes across thousands of samples.

This step includes:
1. **Conversion of `.gct` to `.csv`**, skipping initial metadata rows.
2. **Random sampling of 1000 columns (samples)** to ensure balance with the GDC dataset.
3. Saving the final dataset as `gtx_data.csv` in `data/initial/`.

Like GDC, this step is time-consuming and should be skipped in repeated runs unless data needs to be refreshed.

In [ ]:
# Define paths
gtx_full_gct_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.gct')
gtx_full_csv_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.csv')

##### Step 2.1: Convert GCT to CSV
This block defines the `read_gct()` function, which skips the top two header lines from the `.gct` file and loads the remaining expression matrix into a DataFrame. The `convert_gct_to_csv()` function wraps this logic and saves the result as a temporary `.csv` file.

In [ ]:
def read_gct(file_path):
    with open(file_path, 'r') as f:
        # Skip the first two header lines
        for _ in range(2):
            next(f)
        # Read the rest of the file into a pandas DataFrame
        df = pd.read_csv(f, sep='\t')
    return df

In [ ]:
def convert_gct_to_csv(gct_file, csv_file):
    df = read_gct(gct_file)
    df.to_csv(csv_file, index=False)

In [ ]:
convert_gct_to_csv(gtx_full_gct_file_path, gtx_full_csv_file_path)

##### Step 2.2: Select 1231 samples randomly to make a balanced data sets


In [4]:
gtx_full_csv_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.csv')
gtx_full_df = pd.read_csv(gtx_full_csv_file_path, nrows=5)
columns = gtx_full_df.columns.tolist()

# The first two columns are 'Name' and 'Description', we keep them and sample the rest
gene_info_columns = columns[0:2]
sample_columns = columns[2:]

In [5]:
number_of_samples_to_be_selected = 1231

# Randomly select sample columns from the dataset
chosen_samples_columns = np.random.choice(sample_columns, number_of_samples_to_be_selected, replace=False).tolist()
final_samples_columns = gene_info_columns + chosen_samples_columns

# Load the dataset again but only with the selected columns
gtx_selected_data_df = pd.read_csv(gtx_full_csv_file_path, usecols=final_samples_columns)
gtx_selected_data_df = gtx_selected_data_df.rename(columns={'Name': 'gene_id'})
gtx_selected_data_df = gtx_selected_data_df.drop('Description', axis=1)

# Changing the sample ids to h_xxxx format
num_samples = gtx_selected_data_df.shape[1] - 1
new_sample_names = [f"h_{i:04d}" for i in range(1, num_samples + 1)]
gtx_selected_data_df.columns = [gtx_selected_data_df.columns[0]] + new_sample_names

In [6]:
# Save the sampled data to a new CSV file
gtx_selected_samples_file_path = os.path.join(INITIAL_DIR, 'gtx_data.csv')
gtx_selected_data_df.to_csv(gtx_selected_samples_file_path, index=False)

In [7]:
# Load and display the first few rows of the GTex selected dataset
gtx_data_temp_df = pd.read_csv(gtx_selected_samples_file_path, index_col=0)
print("Shape of initial GTex Data Set", gtx_data_temp_df.shape)
gtx_data_temp_df.head(5)

Shape of initial GTex Data Set (56200, 1231)


,h_0001,h_0002,h_0003,h_0004,h_0005,h_0006,h_0007,h_0008,h_0009,h_0010,...,h_1222,h_1223,h_1224,h_1225,h_1226,h_1227,h_1228,h_1229,h_1230,h_1231
gene_id,,,,,,,,,,,,,,,,,,,,,
ENSG00000223972.5,0.00000,0.000,0.0,0.000,0.0000,0.000,0.03015,0.000,0.000,0.03048,...,0.00000,0.00000,0.000,0.00000,0.000,0.00000,0.04241,0.02613,0.04273,0.0000
ENSG00000227232.5,16.95000,3.082,12.3,3.913,2.9820,3.739,2.74700,3.402,1.907,4.48400,...,3.34300,0.67120,5.317,5.96800,7.555,1.92400,11.10000,5.51100,2.00300,1.0160
ENSG00000278267.1,0.00000,0.000,0.0,0.000,0.0000,0.000,0.00000,0.000,0.000,0.00000,...,0.00000,0.00000,0.000,0.00000,0.000,0.00000,0.00000,0.00000,0.00000,0.4723
ENSG00000243485.5,0.00000,0.000,0.0,0.000,0.0000,0.000,0.00000,0.000,0.000,0.00000,...,0.04577,0.05938,0.000,0.02164,0.000,0.00000,0.00000,0.10430,0.04265,0.0000
ENSG00000237613.2,0.03904,0.000,0.0,0.000,0.0361,0.000,0.00000,0.000,0.000,0.00000,...,0.06503,0.01406,0.000,0.00000,0.000,0.03437,0.06016,0.00000,0.00000,0.0000


##### Step 2.3: Randomly sample 1000 columns (samples)
This block randomly selects 1000 GTEx samples (columns) from the large CSV file produced in the previous step. Only gene identifiers and the selected samples are retained. The resulting matrix is saved to `data/initial/gtx_data.csv`.

In [8]:
gtx_selected_samples_file_path_ = os.path.join(INITIAL_DIR, 'gtx_data.csv')
gtx_data_df = pd.read_csv(gtx_selected_samples_file_path_, nrows=5)
gtx_columns = gtx_data_df.columns.tolist()

# The first two columns are 'Name' and 'Description', we keep them and sample the rest
gtx_gene_info_columns = gtx_columns[0:1]
gtx_samples_columns = gtx_columns[1:]

In [10]:
# Split data into data for model building and training and unseen data for testing
gtx_total_num = len(gtx_samples_columns)         # bc gtex data = 1231
gtx_train_num = 1000
gtx_unseen_num = gtx_total_num - gtx_train_num

In [11]:
# Randomly select 1000 sample columns from the dataset as traiing data
gtx_chosen_columns_for_training = np.random.choice(gtx_samples_columns, gtx_train_num, replace=False).tolist()
gtx_final_columns_for_training = gtx_gene_info_columns + gtx_chosen_columns_for_training

# Randomly select 231 sample columns from the dataset as traiing data
gtx_chosen_columns_for_testing = [col for col in gtx_samples_columns if col not in gtx_chosen_columns_for_training]
gtx_final_columns_for_testing = gtx_gene_info_columns + gtx_chosen_columns_for_testing

# Load the dataset again but only with the selected columns
gtx_train_df = pd.read_csv(gtx_selected_samples_file_path_, usecols=gtx_final_columns_for_training)
gtx_unseen_df = pd.read_csv(gtx_selected_samples_file_path_, usecols=gtx_final_columns_for_testing)

In [12]:
# Save the sampled data to a new CSV file
gtx_training_file_path = os.path.join(INITIAL_DIR, 'gtx_data_training.csv')
gtx_testing_file_path = os.path.join(INITIAL_DIR, 'gtx_data_testing.csv')

gtx_train_df.to_csv(gtx_training_file_path, index=False)
gtx_unseen_df.to_csv(gtx_testing_file_path, index=False)

In [13]:
print("Shape of initial Gtex Training Data Set", gtx_train_df.shape)
gtx_train_df.head(5)

Shape of initial Gtex Training Data Set (56200, 1001)


,gene_id,h_0001,h_0002,h_0003,h_0004,h_0005,h_0006,h_0007,h_0008,h_0009,...,h_1218,h_1219,h_1222,h_1223,h_1224,h_1225,h_1226,h_1227,h_1228,h_1229
0,ENSG00000223972.5,0.00000,0.000,0.0,0.000,0.0000,0.000,0.03015,0.000,0.000,...,0.000,0.00,0.00000,0.00000,0.000,0.00000,0.000,0.00000,0.04241,0.02613
1,ENSG00000227232.5,16.95000,3.082,12.3,3.913,2.9820,3.739,2.74700,3.402,1.907,...,3.726,3.71,3.34300,0.67120,5.317,5.96800,7.555,1.92400,11.10000,5.51100
2,ENSG00000278267.1,0.00000,0.000,0.0,0.000,0.0000,0.000,0.00000,0.000,0.000,...,0.000,0.00,0.00000,0.00000,0.000,0.00000,0.000,0.00000,0.00000,0.00000
3,ENSG00000243485.5,0.00000,0.000,0.0,0.000,0.0000,0.000,0.00000,0.000,0.000,...,0.000,0.00,0.04577,0.05938,0.000,0.02164,0.000,0.00000,0.00000,0.10430
4,ENSG00000237613.2,0.03904,0.000,0.0,0.000,0.0361,0.000,0.00000,0.000,0.000,...,0.000,0.00,0.06503,0.01406,0.000,0.00000,0.000,0.03437,0.06016,0.00000


In [14]:
print("Shape of initial GTex Testing Data Set", gtx_unseen_df.shape)
gtx_unseen_df.head(5)

Shape of initial GTex Testing Data Set (56200, 232)


,gene_id,h_0011,h_0014,h_0023,h_0024,h_0031,h_0034,h_0036,h_0041,h_0045,...,h_1196,h_1199,h_1201,h_1203,h_1210,h_1217,h_1220,h_1221,h_1230,h_1231
0,ENSG00000223972.5,0.000,0.000,0.06823,0.000,0.000,0.0000,0.00000,0.00000,0.000,...,0.000,0.000,0.0276,0.00000,0.04664,0.00000,0.000,0.000,0.04273,0.0000
1,ENSG00000227232.5,6.878,1.254,6.26300,6.763,4.485,0.7543,3.49700,1.68400,3.759,...,5.134,3.316,4.0240,1.66700,6.07200,5.05600,4.808,6.572,2.00300,1.0160
2,ENSG00000278267.1,0.000,0.000,0.00000,0.000,0.000,0.0000,0.00000,0.00000,0.000,...,0.000,0.000,0.0000,0.00000,0.00000,0.00000,0.000,0.000,0.00000,0.4723
3,ENSG00000243485.5,0.000,0.000,0.00000,0.000,0.000,0.0000,0.00000,0.03798,0.000,...,0.000,0.000,0.0000,0.00000,0.00000,0.06862,0.000,0.000,0.04265,0.0000
4,ENSG00000237613.2,0.000,0.000,0.00000,0.000,0.000,0.0000,0.03663,0.00000,0.000,...,0.000,0.000,0.0000,0.02633,0.00000,0.00000,0.000,0.000,0.00000,0.0000


### Step 3: Merge and Label Cancer and Non-Cancer Data

This step merges the GTEx (non-cancer) and GDC (cancer) RNA-Seq datasets and prepares them for machine learning. The goal is to align the gene expression profiles from both sources, assign binary labels, and generate a unified dataset.

This step includes:
1. **Loads**:
   - `gtx_data.csv` and `gdc_data.csv` from `data/initial/`
2. **Aligns gene features** (columns) to keep only shared genes between GTEx and GDC.
3. **Transposes the matrices**:
   - Each **row** becomes a sample,
   - Each **column** is a gene (TPM value).
4. **Assigns labels**:
   - `0` for GTEx (healthy samples),
   - `1` for GDC (cancer samples).
5. **Combines** both datasets into a single feature matrix and a corresponding label vector.

##### Output files (in `data/interim/`):
- `preprocessed_data_features.csv`: Combined matrix of all samples with aligned gene features (samples × genes)
- `preprocessed_data_labels.csv`: Binary labels for each sample (0 = GTEx, 1 = GDC)

These files serve as input to the next steps: statistical feature selection and dimensionality reduction.


In [15]:
gtx_file_path = os.path.join(INITIAL_DIR, 'gtx_data_training.csv')
gdc_file_path = os.path.join(INITIAL_DIR, 'gdc_data_training.csv')

# Load the datasets with headers
gtx__df = pd.read_csv(gtx_file_path, header=0)
gdc__df = pd.read_csv(gdc_file_path, header=0)

In [16]:
# Ensure that the 'gene_id' column is the index for both datasets
gtx__df.set_index('gene_id', inplace=True)
gdc__df.set_index('gene_id', inplace=True)

In [19]:
# Add labels row: 0 for GTEx (healthy) and 1 for GDC (cancer)
gtx_labels = pd.DataFrame([0] * gtx__df.shape[1], index=gtx__df.columns, columns=['label']).transpose()
gdc_labels = pd.DataFrame([1] * gdc__df.shape[1], index=gdc__df.columns, columns=['label']).transpose()

# Concatenate labels and data
gtx__df = pd.concat([gtx_labels, gtx__df])
gdc__df = pd.concat([gdc_labels, gdc__df])

In [ ]:
# Find common genes (rows)
common_genes = gtx__df.index.intersection(gdc__df.index)

# Filter both datasets to keep only the common genes
gtx_df_aligned = gtx__df.loc[common_genes]
gdc_df_aligned = gdc__df.loc[common_genes]


In [ ]:
# Combine both datasets
# Now stack them
combined_data = pd.concat([gtx_df_aligned, gdc_df_aligned], axis=0)

# combined_data = pd.concat([gtx__df, gdc__df], axis=1)

In [ ]:
# Transpose the data to have samples as rows and genes as columns
combined_data = combined_data.transpose()

In [ ]:
# Separate features and labels
labels = combined_data['label']
features = combined_data.drop(columns=['label'])

In [ ]:
# Save preprocessed features and labels to CSV
output_file_prefix = os.path.join(INTERIM_DIR, 'training_data')
features.to_csv(output_file_prefix + '_features.csv', index=False)
labels.to_csv(output_file_prefix + '_labels.csv', index=False)

print("Data Merging Complete")
print("Shape of features:", features.shape)
print("Shape of labels:", labels.shape)